**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher


# Chapter 6: CRSP Stock Data with `farms`

This notebook introduces the `farms` CRSP data loaders. CRSP provides security-level prices, returns, volume, shares outstanding, identifiers, and company information through WRDS.

## Learning objectives

By the end of this notebook, you should be able to:

- Connect to WRDS and identify CRSP securities by PERMNO or ticker.
- Use `farms.load_crsp_data()` to retrieve monthly or daily observations.
- Use `farms.load_all_crsp_data()` to retrieve all securities in a bounded period.
- Apply share-code, market-capitalization, and price screens without look-ahead bias.
- Interpret CRSP prices, returns, volume, and shares outstanding.
- Recognize the optional matching-frequency Ken French market, FF3, or FF5 return data.

## Data access

The examples require a WRDS account with access to CRSP. The `wrds` package is installed separately from `farms`, and the connection cell will prompt for your WRDS credentials and authentication. Do not place credentials in the notebook.

In [ ]:
%pip install -q farms wrds

import farms as fm
import pandas as pd
import wrds

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Establish a WRDS connection

The loader accepts an open `wrds.Connection` object, where `wrds` stands for Wharton Reserch Database.  You will be prompted to enter your WRDS username and password. You may also be prompted to create a `.pgpass` file for future connections. This file allows you to access WRDS databases without entering your credentials each time. Google Colab workspaces are temporary, so a `.pgpass` file created there will be lost when the runtime is reset. Close the connection after the final query with `db.close()`.

In [ ]:
db = wrds.Connection()

## Load CRSP data for selected securities

`farms.load_crsp_data()` loads CRSP monthly or daily stock-file observations for a list of PERMNOs or ticker symbols. Use `YYYY-MM` dates for monthly data and `YYYY-MM-DD` dates for daily data; each range is inclusive. Providing `identifier_type` is recommended because digit-only strings can otherwise be interpreted as PERMNOs.

**Required inputs**
- db — An open wrds.Connection object or compatible database wrapper.
- identifiers — An iterable of PERMNOs or ticker symbols. A single identifier must be passed as a one-item list.
- start_date — Inclusive beginning date: YYYY-MM for monthly data or YYYY-MM-DD for daily data.
- end_date —

**Optional inputs**
- `chunk_size` — Optional; default: `500`. Maximum number of identifiers sent in each SQL query. Smaller values create more queries; larger values create fewer, longer queries. Must be a positive integer.
- identifier_type — How to interpret identifiers. Default: None.
  - "permno" — Treat identifiers as CRSP permanent security numbers.
  - "ticker" — Treat identifiers as ticker symbols.
  - None — Attempt to infer the type automatically.
- frequency — CRSP observation frequency. Default: "monthly".
  - "monthly" — Load CRSP Monthly Stock File data.
  - "daily" — Load CRSP Daily Stock File data.

**Outputs**
- `permno` — CRSP permanent security identifier.
- `permco` — CRSP permanent company identifier.
- `ticker` — Historical CRSP trading ticker.
- `comnam` — Company name.
- `shrcd` — CRSP share code identifying the security type.
- `exchcd` — CRSP exchange code.
- `siccd` — Standard Industrial Classification industry code.
- `prc` — CRSP price. Negative values indicate a bid/ask average was used instead of a closing trade price.
- `ret` — Holding-period return including distributions, in decimal format.
- `retx` — Holding-period return excluding distributions, in decimal format.
- `vol` — Trading volume.
- `shrout` — Shares outstanding, reported by CRSP in thousands.

In [ ]:
monthly = fm.load_crsp_data(
    db,
    identifiers=['AAPL', 'MSFT'],  # Apple and Microsoft tickers
    start_date='2020-01',
    end_date='2020-12',
    identifier_type='ticker',
)

monthly.head()

## Summarize security-level returns

The output from `load_crsp_data()` can contain multiple rows with the same `date`, because each security has its own observation for that period. Grouping by `permno` and `ticker` combines the observations for each security–ticker combination and produces one summary row per security.

In [ ]:
return_summary = (
    monthly.groupby(['permno', 'ticker'])['ret']
    .agg(
        observations='count',
        mean_return='mean',
        volatility='std',
        minimum_return='min',
        maximum_return='max',
    )
    .sort_values('mean_return', ascending=False)
)

return_summary

## Load all securities with possible screens

`farms.load_all_crsp_data()` retrieves all securities in a bounded monthly or daily range.

**Required inputs**
- db — An open `wrds.Connection` object or compatible database wrapper.
- start_date — Inclusive beginning date: YYYY-MM for monthly data or YYYY-MM-DD for daily data.
- end_date — Inclusive ending date using the same format as start_date.

**Optional inputs**
- `frequency` — CRSP observation frequency. Default: "monthly".
  - "monthly" — Load CRSP Monthly Stock File data.
  - "daily" — Load CRSP Daily Stock File data.
- `share_codes` — Restrict results to specific CRSP share codes. Default: None.
- `market_cap_min` — Strict lower bound for beginning-of-period market capitalization, in dollars. Default: None.
- `market_cap_max` — Strict upper bound for beginning-of-period market capitalization, in dollars. Default: None.
- `price_min` — Strict lower bound for beginning-of-period absolute CRSP price. Default: None.
- `price_max` — Strict upper bound for beginning-of-period absolute CRSP price. Default: None.

**Screens**
- The screens are applied row by row, for each security and return.
- Monthly data: each month’s row is screened using that security’s latest observation from the *prior calendar month*.
    - Example: For a March 2009 observation, the screen uses the security’s latest CRSP observation from February 2009, February 27, since February 28 was a Saturday. A stock is included only if it met the price, market-capitalization, or share-code criteria at that prior observation.
- Daily data: each day’s row is screened using that security’s most recent *prior trading observation*.
    - Example: For a January 6, 2020 observation, the screen uses the security’s most recent prior trading observation from January 3, because January 4–5 were the weekend. The January 6 return is included only if the security passed the screen based on its January 3 information.

If no screen inputs are provided, all securities are returned.

In [ ]:
screened_monthly = fm.load_all_crsp_data(
    db,
    start_date='2009-03',
    end_date='2009-06',
    frequency='monthly',
    share_codes=(10, 11),
    price_min=5,
)

print(
    "Number of unique securities:",
    screened_monthly["permno"].nunique()
)

screened_monthly.head()

In [ ]:
# Count the number of securities that pass the screen in each month.
screened_monthly.groupby(level='date').size().rename('number_of_securities').to_frame()

## Load daily data

Set `frequency='daily'` and use `YYYY-MM-DD` dates. Daily screens use the most recent prior CRSP trading observation, which avoids using information from the end of the same day being screened. Daily all-security queries can be large, so keep the date range short while exploring. If requested, `include_factors='market'`, `'ff3'`, or `'ff5'` retrieves matching daily Ken French returns; no factor-loading code is executed in this notebook.

In [ ]:
screened_daily = fm.load_all_crsp_data(
    db,
    start_date='2020-01-02',
    end_date='2020-01-02',
    frequency='daily',
    share_codes=(10, 11),
    price_min=5,
)

print(
    "Number of unique securities:",
    screened_daily["permno"].nunique()
)
screened_daily.head()

In [ ]:
db.close()

## Optional Ken French factor returns

If `include_factors` is requested, the matching-frequency Ken French factor columns are also returned in decimal format. The available options are:

- `None` or `'none'`: return only the CRSP columns; no factor data are merged.
- `'market'`: add `ff_mkt_rf` (the market excess return, `Mkt-RF`) and `ff_rf` (the risk-free return).
- `'ff3'`: add `ff_mkt_rf`, `ff_smb`, `ff_hml`, and `ff_rf`—the market excess return, size, value, and risk-free factors.
- `'ff5'`: add `ff_mkt_rf`, `ff_smb`, `ff_hml`, `ff_rmw`, `ff_cma`, and `ff_rf`—the FF3 factors plus profitability, investment, and the risk-free return.

## Try it

1. Replace the ticker symbols in the first query with PERMNOs such as `[14593, 12079]` and set `identifier_type='permno'`.
2. Compare a monthly screen with and without `share_codes=(10, 11)`.
3. Change `price_min` and `price_max` and compare the number of securities passing the screen.
4. Use a short daily period and compare the daily `DatetimeIndex` with the monthly `PeriodIndex`.
5. Explain why using current-period price or market capitalization for the eligibility screen could introduce look-ahead bias.